In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import chi2, SelectKBest, RFE, f_classif
from sklearn.metrics import accuracy_score, classification_report
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [85]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datasets/wbc.csv')
df.head(2)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN


In [86]:
# check dimension n info cols
print(f"dimension: {df.shape}\n")
# print(df.info())

dimension: (569, 33)



In [87]:
# 1 differentiate usable n unusable var
# X = all feat going to be used to predict | y = result diagnosis going to be predicted

X = df.drop(columns=['id', 'Unnamed: 32', 'diagnosis'], errors='ignore')
y = df['diagnosis']

In [88]:
# 2 encode 'diagnosis' column
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

print("value count: \n")
print(df['diagnosis'].value_counts())
print("\nlist 20 of them: \n")
print(df['diagnosis'].head(20))
print("\nle.classes_ : \n")
print(le.classes_) # Benign = 0, Malignant = 1

value count: 

diagnosis
0    357
1    212
Name: count, dtype: int64

list 20 of them: 

0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    0
Name: diagnosis, dtype: int64

le.classes_ : 

['B' 'M']


In [89]:
# 3 standardization on all columns that posses num val

num_cols = X.columns.tolist()

num_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

preprocess = ColumnTransformer([
    ("num", num_tf, num_cols)
])

std = StandardScaler()
X_std = std.fit_transform(X)
X_std = pd.DataFrame(X_std, columns=X.columns)

print("STD ALL COL: \n")
print(X_std)

print("\nDIAGNOSIS STILL 0 AND 1 \n")
print(y.head())


STD ALL COL: 

     radius_mean  texture_mean  ...  symmetry_worst  fractal_dimension_worst
0       1.097064     -2.073335  ...        2.750622                 1.937015
1       1.829821     -0.353632  ...       -0.243890                 0.281190
2       1.579888      0.456187  ...        1.152255                 0.201391
3      -0.768909      0.253732  ...        6.046041                 4.935010
4       1.750297     -1.151816  ...       -0.868353                -0.397100
..           ...           ...  ...             ...                      ...
564     2.110995      0.721473  ...       -1.360158                -0.709091
565     1.704854      2.085134  ...       -0.531855                -0.973978
566     0.702284      2.045574  ...       -1.104549                -0.318409
567     1.838341      2.336457  ...        1.919083                 2.219635
568    -1.808401      1.221792  ...       -0.048138                -0.751207

[569 rows x 30 columns]

DIAGNOSIS STILL 0 AND 1 

0    M
1 

In [90]:
# 4 feat seelction use SelectKBest
# 5 testing using logistig regression model
# ACCURACY = 0.9649 (96.5%)

selector_filter = SelectKBest(score_func=f_classif, k=5)

pipe_filter = Pipeline([
    ("prep", preprocess),
    ("sel", selector_filter),
    ("clf", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

pipe_filter.fit(X_train, y_train)
pred = pipe_filter.predict(X_test)

print("\n=== FILTER (ANOVA) + LR ===\n")
print("accuracy: ", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


=== FILTER (ANOVA) + LR ===

accuracy:  0.9649122807017544
              precision    recall  f1-score   support

           B       0.96      0.99      0.97        72
           M       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [91]:
# 6 employ a model pipeline

feat_names = pipe_filter.named_steps["prep"].get_feature_names_out()
sel = pipe_filter.named_steps["sel"]
mask = sel.get_support()

selected_names = feat_names[mask]
selected_scores = sel.scores_[mask]

top = sorted(zip(selected_names, selected_scores), key=lambda t: t[1], reverse=True)[:10]

print("top feature: ",top)

top feature:  [('num__concave points_worst', np.float64(733.7249325739482)), ('num__perimeter_worst', np.float64(717.2464871041376)), ('num__radius_worst', np.float64(692.861395260564)), ('num__concave points_mean', np.float64(684.5268452011383)), ('num__perimeter_mean', np.float64(548.4132358502825))]


In [92]:
# 7 how many optimal features can be used?

print("top feature: ",top)

# TOP FEATURE
# 1. concave points_worst (ANOVA: 733.72)
# 2. perimiter_worst (ANOVA: 717,25)
# 3. radius_worst (ANOVA: 692.86)
# 4. concave points_mean (ANOVA: 684.53)
# 5. perimiter_mean (ANOVA: 548.41)

top feature:  [('num__concave points_worst', np.float64(733.7249325739482)), ('num__perimeter_worst', np.float64(717.2464871041376)), ('num__radius_worst', np.float64(692.861395260564)), ('num__concave points_mean', np.float64(684.5268452011383)), ('num__perimeter_mean', np.float64(548.4132358502825))]
